<a href="https://colab.research.google.com/github/Ivan07cabrera/Experimentos_tarea3/blob/main/Experimentos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install wandb optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 6.8 MB/s eta 0:00:00


In [ ]:
# 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

In [3]:
!wandb login

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: cabreratrinidadangelivan (cabreratrinidadangelivan-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
#Librerias
import optuna
import wandb

from wandb.integration.keras import WandbMetricsLogger

import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input


In [5]:
# iniciar proyecto
wandb.init(project="experimentos-optuma-ultv")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: cabreratrinidadangelivan (cabreratrinidadangelivan-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
# cargar datos
dataset = mnist.load_data()
(x_train, y_train), (x_test, y_test) = dataset

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
# normalizar los datos
x_train = x_train.astype("float32")/255
x_test = x_test.astype("float32")/255

In [8]:
# one hot encoding
num_classes = 10
y_trainc = keras.utils.to_categorical(y_train, num_classes)
y_testc = keras.utils.to_categorical(y_test, num_classes)


In [9]:
# función objetivo para Optuna
def objective(trial):

    wandb.init(
        project="experimentos-optuma-ultversion",
        reinit=True
    )


    model = Sequential()
    model.add(Input(shape=(28,28)))
    model.add(Flatten())


    # eleccion del número de capas (entre 1 y 4)
    n_capas = trial.suggest_int("n_capas", 1, 4)

    for i in range(n_capas):

        units = trial.suggest_int(f"units_l{i}", 32, 512)

        activation = trial.suggest_categorical(
            f"activation_l{i}",
            ["tanh", "sigmoid", "relu","softplus"]  #funciones de activación a utilizar
        )

        model.add(Dense(units, activation=activation))

    model.add(Dense(10, activation="softmax"))      #funcion de activación ultima capa

    optimizer_name = trial.suggest_categorical(
        "optimizer",
        ["adam", "sgd", "rmsprop"]     #optimizadores a escoger
    )

    model.compile(
    optimizer=optimizer_name,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]

    )


   #Tamaño de batch
    batch_size = trial.suggest_categorical(
        "batch_size",
        [32, 64, 128]
    )

     # guardar parámetros en wandb
    wandb.config.update({
        "n_capas": n_capas,
        "optimizer": optimizer_name,
        "batch_size": batch_size
    })

    history = model.fit(
        x_train,
        y_train,
        validation_data=(x_test, y_test),
        epochs=10,
        batch_size=batch_size,
        verbose=1,
        callbacks=[WandbMetricsLogger()]
    )

    score = model.evaluate(x_test, y_test, verbose=0)

    wandb.finish()

    return score[1]

In [10]:
# ejecutar experimentos
study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=10)   #numero de experimentos


[I 2026-05-07 04:05:44,248] A new study created in memory with name: no-name-b3ccd9c0-cf98-4140-914a-5ba9c825ef91


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9177 - loss: 0.2727 - val_accuracy: 0.9564 - val_loss: 0.1481
Epoch 2/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9625 - loss: 0.1254 - val_accuracy: 0.9644 - val_loss: 0.1136
Epoch 3/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - accuracy: 0.9730 - loss: 0.0868 - val_accuracy: 0.9694 - val_loss: 0.1011
Epoch 4/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.9812 - loss: 0.0623 - val_accuracy: 0.9727 - val_loss: 0.0861
Epoch 5/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.9843 - loss: 0.0497 - val_accuracy: 0.9745 - val_loss: 0.0799
Epoch 6/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.9884 - loss: 0.0375 - val_accuracy: 0.9756 - val_loss: 0.0823
Epoch 7/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 7s 14ms/step - accuracy: 0.9909 - loss: 0.0289 - val_accuracy: 0.9769 - val_loss: 0.0703
Epoch 8/10
469/469 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.9930 - loss: 0.0229 - val_a

epoch/accuracy,▁▅▆▇▇▇████
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▂▂▂▁▁▁▁
epoch/val_accuracy,▁▄▅▆▇▇██▇█
epoch/val_loss,█▅▄▂▂▂▁▂▂▂
epoch/accuracy,0.99462
epoch/epoch,9
epoch/learning_rate,0.001
epoch/loss,0.01735
epoch/val_accuracy,0.9768


[I 2026-05-07 04:07:11,273] Trial 0 finished with value: 0.9768000245094299 and parameters: {'n_capas': 3, 'units_l0': 226, 'activation_l0': 'tanh', 'units_l1': 344, 'activation_l1': 'tanh', 'units_l2': 220, 'activation_l2': 'tanh', 'optimizer': 'adam', 'batch_size': 128}. Best is trial 0 with value: 0.9768000245094299.


...

epoch/accuracy,▁▅▆▆▇▇▇███
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▃▂▂▂▁▁▁
epoch/val_accuracy,▁▃▅▆▆▇▇███
epoch/val_loss,█▆▄▃▃▂▂▁▁▁
epoch/accuracy,0.98148
epoch/epoch,9
epoch/learning_rate,0.001
epoch/loss,0.06348
epoch/val_accuracy,0.9744


[I 2026-05-07 04:24:17,773] Trial 8 finished with value: 0.974399983882904 and parameters: {'n_capas': 1, 'units_l0': 105, 'activation_l0': 'softplus', 'optimizer': 'adam', 'batch_size': 128}. Best is trial 1 with value: 0.9818999767303467.


Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.8245 - loss: 0.6451 - val_accuracy: 0.8940 - val_loss: 0.3647
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9047 - loss: 0.3240 - val_accuracy: 0.9165 - val_loss: 0.2842
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9174 - loss: 0.2805 - val_accuracy: 0.9277 - val_loss: 0.2536
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9261 - loss: 0.2509 - val_accuracy: 0.9320 - val_loss: 0.2287
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9331 - loss: 0.2266 - val_accuracy: 0.9359 - val_loss: 0.2154
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9388 - loss: 0.2062 - val_accuracy: 0.9417 - val_loss: 0.1988
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9445 - loss: 0.1886 - val_accuracy: 0.9456 - val_loss: 0.1812
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9489 - loss: 0

epoch/accuracy,▁▅▆▆▇▇▇███
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▃▃▂▂▂▂▁▁▁
epoch/val_accuracy,▁▄▅▆▆▇▇███
epoch/val_loss,█▅▄▃▃▃▂▂▁▁
epoch/accuracy,0.95535
epoch/epoch,9
epoch/learning_rate,0.01
epoch/loss,0.14853
epoch/val_accuracy,0.953


[I 2026-05-07 04:26:49,952] Trial 9 finished with value: 0.953000009059906 and parameters: {'n_capas': 3, 'units_l0': 443, 'activation_l0': 'softplus', 'units_l1': 142, 'activation_l1': 'tanh', 'units_l2': 215, 'activation_l2': 'tanh', 'optimizer': 'sgd', 'batch_size': 32}. Best is trial 1 with value: 0.9818999767303467.
